In [ ]:
print('hello money')

In [ ]:
# ------------- configure dataframe printing layout 

import pandas as pd
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
#------------------------------------------------

In [ ]:
# FILE PATH and SQL QUERY

import numpy as np
import duckdb
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# file path
folder = Path.cwd() / '_inputs'/'dayly_klines'
# folder = Path.cwd() / '_inputs'/'hourly_klines'
# folder = Path.cwd() / '_inputs'/'_5m_klines'

# access the files
sorted_pqt_files = sorted(folder.glob('*parquet'))
# file list
pqt_file = [pqt.as_posix() for pqt in sorted_pqt_files]

# duckdb query for the data
conn = duckdb.connect()

qry = f"""--sql
        select 
            --cast for ms to readeable time
            --epoch_ms(CAST(open_time as bigint)) as open_time,
            open_time,
            open,
            high,
            low,
            close,
            close_time
        from read_parquet({pqt_file})    
        """

df = conn.sql(qry).df()
# df = conn.execute(query).df()
conn.close()

df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')
df["sma25"] = df['close'].rolling(25).mean()

# df.tail()

In [ ]:
# ADDING THE PIVOT POINTS TO THE TABLE
# ----------------------#option 1 with wick swings for stoploss management ---------------
left_len =10
right_len = 10
window_size = left_len + right_len + 1
#Rolling max/min centered on the target candle
rolling_max = df['high'].rolling(window= window_size, center=True).max()
rolling_min = df['low'].rolling(window= window_size, center= True).min()
# Indentify pivot points (hing must equal max of the window and low must equal min of the window )
df['pivot_high'] = df['high'].where(df['high']==rolling_max, None)
df['pivot_low'] = df['low'].where(df['low']==rolling_min, None)
# Combined column showing the price level at the swing point, or None otherwise
df['swing_point_wcks'] = df['pivot_high'].combine_first(df['pivot_low'])

# ----------------------#option 2 True candle body change that is fillable ---------------
#  Calculate true top and bottom of the candle body
df['body_top'] = df[['open', 'close']].max(axis=1)
df['body_bottom'] = df[['open', 'close']].min(axis=1)
# Rolling max/min centered on the candle BODY
rolling_body_max = df['body_top'].rolling(window=window_size, center=True).max()
rolling_body_min = df['body_bottom'].rolling(window=window_size, center=True).min()
#  Identify pivot points using body boundaries
df['pivot_high'] = df['body_top'].where(df['body_top'] == rolling_body_max, None)
df['pivot_low'] = df['body_bottom'].where(df['body_bottom'] == rolling_body_min, None)
# 4. Combined swing point column
df['swing_point_bdy1'] = df['pivot_high'].combine_first(df['pivot_low'])

# --------------------Option 3: Structural Pivot Detection with Body-Level Price Extraction

# 1. Identify structural pivot candles using High/Low extremes
rolling_max = df['high'].rolling(window=window_size, center=True).max()
rolling_min = df['low'].rolling(window=window_size, center=True).min()

df['pivot_h_raw'] = df['high'].where(df['high'] == rolling_max, None)
df['pivot_l_raw'] = df['low'].where(df['low'] == rolling_min, None)

# 2. Extract true Body Top and Body Bottom levels across all candles
body_top = df[['open', 'close']].max(axis=1)
body_bottom = df[['open', 'close']].min(axis=1)

# 3. Read off Body Top/Bottom ONLY on candles identified as structural pivots
df['pivot_high'] = np.where(df['pivot_h_raw'].notna(), body_top, None)
df['pivot_low'] = np.where(df['pivot_l_raw'].notna(), body_bottom, None)

# 4. Combine into final swing point column (body-based price level)
df['swing_point'] = df['pivot_high'].combine_first(df['pivot_low'])

# Clean up temporary raw detector columns if desired
df = df.drop(columns=['pivot_h_raw', 'pivot_l_raw'])



In [ ]:
# DF for PIVOT POINTS ONLY

pv_df = df[df['swing_point'].notna()]

# print(pv_df.to_string())

In [ ]:
# SWING PIVOT POINTS TABLE

# 1. filter out only the candles that formed valid pivots
pivots = df[df['swing_point'].notna()].copy()
# 2. determine pivot type (high or low ) for each pivot point
pivots['type']= np.where(pivots['pivot_high'].notna(), 'high', 'low')
# 3. Reset index to calculate exact bar index distance between pivots
pivots = pivots.reset_index().rename(columns={'index':'orig_bar_index'})

# BUILD THE SWING POINTS TABLE (spt)
spt = pd.DataFrame()

# Start & End attributes per move (shift by -1 to pair consecutive pivots)
spt['swing_id'] = range(1, len(pivots))
spt['swing_start_time'] = pivots['open_time'][:-1].values
spt['swing_start_price']   = pivots['swing_point'][:-1].values
# spt['swing_end_time'] = pivots['open_time'][1:].values
spt['swing_end_time'] = pivots['close_time'][1:].values   
spt['swing_end_price']   = pivots['swing_point'][1:].values

#bars elapsed between start and end of the swing move
spt['bars'] = (pivots['orig_bar_index'][1:].values - pivots['orig_bar_index'][:-1].values)
#percentage price change
spt['price_change_pct'] =  ((spt['swing_end_price']- spt['swing_start_price'])/spt['swing_start_price'])*100
#duration of the swing move
spt['duration']= (spt['swing_end_time']- spt['swing_start_time'])
#swing direction ; up or down
spt['direction']= np.where(spt['swing_end_price']>spt['swing_start_price'], 'up', 'down')

#Trend classification (bullish, bearish , ranging)
conditions =[spt['price_change_pct']>1.0, # bullish when change greater than 1%
             spt['price_change_pct']<-1.0 # bearish   
            ]
choices =['bullish', 'bear']

spt['trend']= np.select(conditions, choices, default='range')


# print(pivots.head())
# print(spt[150:].to_string())

print(spt.head())


In [ ]:
# PLOTTING THE CANDLE GRAPH

fig = go.Figure()

fig = go.Figure(data=[
    go.Candlestick(
        x=df['open_time'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='Price',
        # bullish candles
        increasing_line_color='white',
        increasing_fillcolor='rgba(0, 0, 0, 0)',
        increasing_line_width=1,
        # bearish candles
        decreasing_line_color='#0677d4',
        decreasing_fillcolor='rgba(0, 0, 0, 0)',
        decreasing_line_width=1
    )
])
# -------------------------------------------- sma line
fig.add_trace(go.Scatter(x=df.open_time, y=df.sma25,
              mode='lines', line=dict(color='purple', width=1)))
# ---------------------------------------swing points line
fig.add_trace(go.Scatter(x=df.open_time, y=df.swing_point,
                         connectgaps= True,
              mode='lines', line=dict(color='green', width=1)))
# ----------------------------------------------
fig.update_layout(title='sol 2026 jan to june  day timeframe',
                  template='plotly_dark',
                  xaxis_rangeslider_visible=False,
                  hovermode='x unified',
                  showlegend=False,
                  height=600
                  )

# Add Pivot High Markers
pivot_highs = df.dropna(subset=['pivot_high'])
fig.add_trace(go.Scatter(
    x=pivot_highs['open_time'],
    y=pivot_highs['pivot_high'],
    mode='markers+text',
    marker=dict(symbol='triangle-down', size=10, color='red'),
    text=pivot_highs['pivot_high'],
    textposition='top center',
    name='Pivot High'
))

# Add Pivot Low Markers
pivot_lows = df.dropna(subset=['pivot_low'])
fig.add_trace(go.Scatter(
    x=pivot_lows['open_time'],
    y=pivot_lows['pivot_low'],
    mode='markers+text',
    marker=dict(symbol='triangle-up', size=10, color='green'),
    text=pivot_lows['pivot_low'],
    textposition='bottom center',
    name='Pivot Low'
))